In [1]:
# ============================================================
# 【多GPU优化版】全部代码块（ipynb格式，按 Code one ~ Code nineteen 排列）
# 优化策略：
#   - GAT/GATv2/所有模型 → 全部上GPU
#   - 参数搜索：多进程并行，每个GPU独立跑一个参数组合
#   - 使用 torch.multiprocessing + GPU分配实现真正并行
# ============================================================

# ============================================================
# Code one: 初始化环境（多GPU版）
# ============================================================
import os
import gc
import pickle
import json
import queue
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, GATv2Conv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.neighbors import kneighbors_graph
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support,
    roc_auc_score,
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from itertools import product
from datetime import datetime
from openpyxl import load_workbook

warnings.filterwarnings("ignore")

# ── 随机种子 ──────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

# ── 检测所有可用GPU ────────────────────────────────────────
NUM_GPUS = torch.cuda.device_count()
print(f"检测到 {NUM_GPUS} 张GPU")
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    free_mem = torch.cuda.mem_get_info(i)[0] / 1024**3
    print(
        f"  GPU {i}: {props.name} | 总显存: {props.total_memory/1024**3:.1f}GB | 可用: {free_mem:.1f}GB"
    )

# ── 设备列表（优先GPU，无GPU退回CPU）──────────────────────
if NUM_GPUS > 0:
    DEVICES = [torch.device(f"cuda:{i}") for i in range(NUM_GPUS)]
    # 并行任务数 = GPU数量（每GPU同时跑1个任务，避免OOM）
    MAX_PARALLEL = NUM_GPUS
else:
    DEVICES = [torch.device("cpu")]
    MAX_PARALLEL = 1
    print("警告：未检测到GPU，将使用CPU运行")

device_GPU = DEVICES[0]  # 向后兼容（单模型测试用）
device_CPU = torch.device("cpu")

print(f"\n并行任务数: {MAX_PARALLEL}")
print(f"主设备: {device_GPU}")

# ── multiprocessing启动方式（必须在最顶层设置）────────────
mp.set_start_method("spawn", force=True)

# ── PyTorch环境变量优化 ───────────────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
cpu_cores = os.cpu_count()
torch.set_num_threads(max(1, cpu_cores // MAX_PARALLEL))  # 多进程时每进程分配线程




/home/ailab/miniconda3/envs/GNNTrain/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


检测到 8 张GPU
  GPU 0: NVIDIA GeForce RTX 3080 | 总显存: 19.7GB | 可用: 19.5GB
  GPU 1: NVIDIA GeForce RTX 3080 | 总显存: 19.7GB | 可用: 10.5GB
  GPU 2: NVIDIA GeForce RTX 3080 | 总显存: 19.7GB | 可用: 19.5GB
  GPU 3: NVIDIA GeForce RTX 3080 | 总显存: 19.7GB | 可用: 19.5GB
  GPU 4: NVIDIA GeForce RTX 3080 | 总显存: 19.7GB | 可用: 19.5GB
  GPU 5: NVIDIA GeForce RTX 3080 | 总显存: 19.7GB | 可用: 19.5GB
  GPU 6: NVIDIA GeForce RTX 3080 | 总显存: 19.7GB | 可用: 19.5GB
  GPU 7: NVIDIA GeForce RTX 3080 | 总显存: 19.7GB | 可用: 19.5GB

并行任务数: 8
主设备: cuda:0


In [2]:
# ============================================================
# Code two: 目录管理（不变）
# ============================================================
print("\n" + "=" * 60)
print("0. 初始化实验目录结构")
print("=" * 60)

BASE_DIR = "./"
MODELS_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
FIGURES_DIR = os.path.join(BASE_DIR, "figures")
CURVES_DIR = os.path.join(FIGURES_DIR, "training_curves")
CONFUSION_DIR = os.path.join(FIGURES_DIR, "confusion_matrices")

for d in [BASE_DIR, MODELS_DIR, RESULTS_DIR, FIGURES_DIR, CURVES_DIR, CONFUSION_DIR]:
    os.makedirs(d, exist_ok=True)

MODEL_DIRS = {
    "MLP": os.path.join(MODELS_DIR, "MLP"),
    "GCN": os.path.join(MODELS_DIR, "GCN"),
    "GraphSAGE": os.path.join(MODELS_DIR, "GraphSAGE"),
    "GAT": os.path.join(MODELS_DIR, "GAT"),
    "GATv2": os.path.join(MODELS_DIR, "GATv2"),
}
for d in MODEL_DIRS.values():
    os.makedirs(d, exist_ok=True)

EXCEL_PATHS = {
    "Xenium": "./results/exports/xenium_for_gnn.csv",
    "MLP": os.path.join(MODEL_DIRS["MLP"], "MLP_results.xlsx"),
    "GCN": os.path.join(MODEL_DIRS["GCN"], "GCN_results.xlsx"),
    "GraphSAGE": os.path.join(MODEL_DIRS["GraphSAGE"], "GraphSAGE_results.xlsx"),
    "GAT": os.path.join(MODEL_DIRS["GAT"], "GAT_results.xlsx"),
    "GATv2": os.path.join(MODEL_DIRS["GATv2"], "GATv2_results.xlsx"),
}

FIGURE_PATHS = {
    "validation_comparison": os.path.join(
        FIGURES_DIR, "model_validation_comparison.png"
    ),
    "test_comparison": os.path.join(FIGURES_DIR, "model_test_accuracy_comparison.png"),
    "f1_comparison": os.path.join(FIGURES_DIR, "model_f1_comparison.png"),
    "parameter_analysis": os.path.join(FIGURES_DIR, "parameter_analysis.png"),
}

print("✓ 目录初始化完成")


0. 初始化实验目录结构
✓ 目录初始化完成


In [3]:
# ============================================================
# Code three: 辅助函数（不变）
# ============================================================


def save_model_and_results(model, config, model_name, training_history=None):
    model_filename = (
        f"{model_name}_h{config['hidden_dim']}_d{config['dropout']}"
        f"_l{config['num_layers']}"
    )
    if "heads" in config:
        model_filename += f"_heads{config['heads']}"
    if "use_skip" in config:
        model_filename += f"_skip{config['use_skip']}"
    model_filename += ".pt"

    model_path = os.path.join(MODEL_DIRS[model_name], model_filename)

    save_dict = {
        "model_state_dict": model.state_dict(),
        "config": config,
        "best_val_acc": config["best_val_acc"],
        "test_acc": config["test_acc"],
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }
    if training_history:
        save_dict["training_history"] = training_history
    torch.save(save_dict, model_path)

    result_row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "hidden_dim": config["hidden_dim"],
        "dropout": config["dropout"],
        "num_layers": config["num_layers"],
        "best_val_acc": config["best_val_acc"],
        "test_acc": config["test_acc"],
        "test_f1_macro": config["test_f1_macro"],
        "test_f1_weighted": config["test_f1_weighted"],
        "test_precision_macro": config.get("test_precision_macro", 0),
        "test_recall_macro": config.get("test_recall_macro", 0),
        "num_params": config.get("num_params", 0),
        "model_file": model_filename,
    }
    if "heads" in config:
        result_row["heads"] = config["heads"]
    if "use_skip" in config:
        result_row["use_skip"] = config["use_skip"]

    excel_path = EXCEL_PATHS[model_name]
    if os.path.exists(excel_path):
        try:
            existing_df = pd.read_excel(excel_path)
            results_df = pd.concat(
                [existing_df, pd.DataFrame([result_row])], ignore_index=True
            )
        except:
            results_df = pd.DataFrame([result_row])
    else:
        results_df = pd.DataFrame([result_row])

    results_df = results_df.sort_values("best_val_acc", ascending=False)
    results_df.to_excel(excel_path, index=False)
    print(f"  ✓ 模型已保存: {model_path}")
    return model_path


def save_detailed_results(
    model_name, config, training_history, test_true, test_pred, class_names
):
    precision, recall, f1, support = precision_recall_fscore_support(
        test_true, test_pred, average=None
    )
    detailed_results = {
        "model_name": model_name,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "config": {
            k: v
            for k, v in config.items()
            if k
            not in ["test_true", "test_pred", "test_pred_labels", "test_true_labels"]
        },
        "best_val_acc": float(config["best_val_acc"]),
        "test_metrics": {
            "accuracy": float(config["test_acc"]),
            "f1_macro": float(config["test_f1_macro"]),
            "f1_weighted": float(config["test_f1_weighted"]),
            "precision_macro": float(config.get("test_precision_macro", 0)),
            "recall_macro": float(config.get("test_recall_macro", 0)),
        },
        "per_class_metrics": {},
    }
    for i in range(len(class_names)):
        detailed_results["per_class_metrics"][f"class_{i}"] = {
            "class_name": str(class_names[i]),
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "support": int(support[i]),
        }
    if training_history:
        detailed_results["training_history_summary"] = {
            "final_train_loss": (
                float(training_history["train_losses"][-1])
                if training_history["train_losses"]
                else None
            ),
            "final_val_loss": (
                float(training_history["val_losses"][-1])
                if training_history["val_losses"]
                else None
            ),
            "best_val_acc": float(config["best_val_acc"]),
            "best_val_loss": (
                float(min(training_history["val_losses"]))
                if training_history["val_losses"]
                else None
            ),
            "epochs_trained": len(training_history["train_losses"]),
        }
    param_str = f"h{config['hidden_dim']}_d{config['dropout']}_l{config['num_layers']}"
    if "heads" in config:
        param_str += f"_heads{config['heads']}"
    filename = f"{model_name}_{param_str}.json"
    save_path = os.path.join(MODEL_DIRS[model_name], filename)
    with open(save_path, "w") as f:
        json.dump(detailed_results, f, indent=2)
    return save_path

In [4]:
# ============================================================
# Code four ~ eight: 数据加载、特征提取、建图、数据集划分（不变）
# ============================================================

# ── 1. 加载数据 ───────────────────────────────────────────
print("\n" + "=" * 60)
print("1. 加载数据")
print("=" * 60)
df = pd.read_csv(EXCEL_PATHS["Xenium"])
df = df.dropna()
print(f"数据形状（去除缺失值后）: {df.shape}")

# ── 2. 提取特征和标签 ──────────────────────────────────────
print("\n" + "=" * 60)
print("2. 提取特征和标签")
print("=" * 60)
coords = df[["x_centroid", "y_centroid"]].values
pca_cols = [col for col in df.columns if col.startswith("PC_")]
features = df[pca_cols].values
scaler = StandardScaler()
features = scaler.fit_transform(features)
labels_raw = df["predicted_cell_type"].values
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels_raw)
num_classes = len(label_encoder.classes_)
class_names = label_encoder.classes_.tolist()
print(f"PCA特征维度: {features.shape} | 类别数: {num_classes}")

# ── 3. 建图 ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("3. 构建空间邻接图")
print("=" * 60)


def build_spatial_graph(coords, k=10):
    adj_matrix = kneighbors_graph(
        coords, n_neighbors=k, mode="connectivity", include_self=False
    )
    edge_index, _ = from_scipy_sparse_matrix(adj_matrix)
    return edge_index


DEFAULT_K = 5
edge_index = build_spatial_graph(coords, k=DEFAULT_K)
print(f"使用 k={DEFAULT_K} | 边数量: {edge_index.shape[1]}")

# ── 4. 创建PyG数据对象 ────────────────────────────────────
x = torch.FloatTensor(features)
y = torch.LongTensor(labels)
edge_index = edge_index.to(torch.long)
data = Data(x=x, edge_index=edge_index, y=y)
print(
    f"节点数: {data.num_nodes} | 特征维度: {data.num_features} | 边数: {data.num_edges}"
)

# ── 5. 划分数据集 ─────────────────────────────────────────
indices = np.arange(data.num_nodes)
train_val_idx, test_idx = train_test_split(
    indices, test_size=0.2, stratify=labels, random_state=42
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.25, stratify=labels[train_val_idx], random_state=42
)

train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
val_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask
print(
    f"训练集: {train_mask.sum().item()} | 验证集: {val_mask.sum().item()} | 测试集: {test_mask.sum().item()}"
)


1. 加载数据
数据形状（去除缺失值后）: (406611, 54)

2. 提取特征和标签
PCA特征维度: (406611, 50) | 类别数: 16

3. 构建空间邻接图
使用 k=5 | 边数量: 2033055
节点数: 406611 | 特征维度: 50 | 边数: 2033055
训练集: 243966 | 验证集: 81322 | 测试集: 81323


In [5]:
# ============================================================
# Code nine: 模型定义（不变，直接复用原始定义）
# ============================================================
print("\n" + "=" * 60)
print("6. 定义图神经网络模型")
print("=" * 60)


class MLPWithDropout(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.3, num_layers=2):
        super().__init__()
        self.layers = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.layers.append(nn.Linear(in_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 2):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.layers.append(nn.Linear(hidden_dim, out_dim))
        self.dropout = dropout

    def forward(self, data):
        x = data.x
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return F.log_softmax(self.layers[-1](x), dim=1)


class GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.3, num_layers=2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(GCNConv(in_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.convs.append(GCNConv(hidden_dim, out_dim))
        self.dropout = dropout
        self.skip_proj = nn.Linear(in_dim, hidden_dim) if num_layers >= 2 else None

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        identity = x
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            if i == 0 and self.skip_proj is not None:
                x = x + self.skip_proj(identity)
        return F.log_softmax(self.convs[-1](x, edge_index), dim=1)


class GraphSAGEWithNorm(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.3, num_layers=2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(SAGEConv(in_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.convs.append(SAGEConv(hidden_dim, out_dim))
        self.dropout = dropout

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return F.log_softmax(self.convs[-1](x, edge_index), dim=1)


class GAT(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim,
        out_dim,
        heads=8,
        dropout=0.3,
        num_layers=2,
        use_skip=True,
    ):
        super().__init__()
        self.num_layers = num_layers
        self.use_skip = use_skip
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(
            GATConv(in_dim, hidden_dim, heads=heads, dropout=dropout, concat=True)
        )
        self.bns.append(nn.BatchNorm1d(hidden_dim * heads))
        for _ in range(num_layers - 2):
            self.convs.append(
                GATConv(
                    hidden_dim * heads,
                    hidden_dim,
                    heads=heads,
                    dropout=dropout,
                    concat=True,
                )
            )
            self.bns.append(nn.BatchNorm1d(hidden_dim * heads))
        self.convs.append(
            GATConv(hidden_dim * heads, out_dim, heads=1, dropout=dropout, concat=False)
        )
        self.dropout = dropout
        if use_skip and num_layers >= 2:
            self.skip_proj = nn.Linear(in_dim, hidden_dim * heads)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        identity = (
            self.skip_proj(x) if (self.use_skip and self.num_layers >= 2) else None
        )
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = self.bns[i](x)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            if identity is not None and i == 0:
                x = x + identity
        return F.log_softmax(self.convs[-1](x, edge_index), dim=1)


class GATv2Model(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=8, dropout=0.3, num_layers=2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(
            GATv2Conv(in_dim, hidden_dim, heads=heads, dropout=dropout, concat=True)
        )
        self.bns.append(nn.BatchNorm1d(hidden_dim * heads))
        for _ in range(num_layers - 2):
            self.convs.append(
                GATv2Conv(
                    hidden_dim * heads,
                    hidden_dim,
                    heads=heads,
                    dropout=dropout,
                    concat=True,
                )
            )
            self.bns.append(nn.BatchNorm1d(hidden_dim * heads))
        self.convs.append(
            GATv2Conv(
                hidden_dim * heads, out_dim, heads=1, dropout=dropout, concat=False
            )
        )
        self.dropout = dropout

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = self.bns[i](x)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return F.log_softmax(self.convs[-1](x, edge_index), dim=1)


print("所有模型定义完成")


6. 定义图神经网络模型
所有模型定义完成


In [6]:
# ============================================================
# Code ten: 训练与评估函数（多GPU自适应版）
# ============================================================
print("\n" + "=" * 60)
print("7. 定义训练和评估函数（多GPU版）")
print("=" * 60)


def clear_memory(device=None):
    if device is not None and device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
    gc.collect()


def train_model(model, data, device, optimizer, epochs=500, patience=50, verbose=True):
    """
    单卡训练函数（device可以是任意GPU或CPU）
    多GPU并行由外层 run_parallel_experiments 调度
    """
    data = data.to(device)
    model = model.to(device)

    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    best_val_acc = 0
    best_val_loss = float("inf")
    best_model_state = None
    patience_counter = 0

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    val_interval = 1  # GPU够快，每epoch都验证

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        if use_amp:
            with torch.amp.autocast("cuda"):
                out = model(data)
                loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(data)
            loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            pred = out.argmax(dim=1)
            train_acc = (
                (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            )
        train_losses.append(loss.item())
        train_accs.append(train_acc)

        if (epoch + 1) % val_interval == 0:
            model.eval()
            with torch.no_grad():
                out_val = model(data)
                val_loss = F.nll_loss(out_val[data.val_mask], data.y[data.val_mask])
                pred_val = out_val.argmax(dim=1)
                val_acc = (
                    (pred_val[data.val_mask] == data.y[data.val_mask])
                    .float()
                    .mean()
                    .item()
                )
            val_losses.append(val_loss.item())
            val_accs.append(val_acc)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_val_loss = val_loss.item()
                best_model_state = {
                    k: v.cpu().clone() for k, v in model.state_dict().items()
                }
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= patience:
                if verbose:
                    print(f"  早停于 epoch {epoch+1}")
                break

            if verbose and (epoch + 1) % 50 == 0:
                print(
                    f"  Epoch {epoch+1:4d} | Loss: {loss.item():.4f} | "
                    f"Train: {train_acc:.4f} | Val: {val_acc:.4f}"
                )

        if use_amp and (epoch + 1) % 100 == 0:
            clear_memory(device)

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model, train_losses, val_losses, train_accs, val_accs, best_val_acc


def evaluate_model(model, data, device):
    model.eval()
    data = data.to(device)
    with torch.no_grad():
        out = model(data)
        pred = out.argmax(dim=1)
        test_pred = pred[data.test_mask].cpu().numpy()
        test_true = data.y[data.test_mask].cpu().numpy()

    test_acc = accuracy_score(test_true, test_pred)
    test_f1_macro = f1_score(test_true, test_pred, average="macro")
    test_f1_weighted = f1_score(test_true, test_pred, average="weighted")
    precision, recall, _, _ = precision_recall_fscore_support(
        test_true, test_pred, average="macro"
    )

    return {
        "test_acc": float(test_acc),
        "test_f1_macro": float(test_f1_macro),
        "test_f1_weighted": float(test_f1_weighted),
        "test_precision_macro": float(precision),
        "test_recall_macro": float(recall),
        "test_pred": test_pred.tolist(),
        "test_true": test_true.tolist(),
    }


# ============================================================
# ★★★ 多GPU并行核心函数 ★★★
# ============================================================


def _worker_process(
    gpu_id, task_queue, result_queue, data_dict, model_name, class_names_list
):
    """
    子进程工作函数：从task_queue取任务，在指定GPU上训练，结果写入result_queue
    每个GPU对应一个独立进程，持续取任务直到收到None信号
    """
    import os, torch, gc
    import torch.nn.functional as F
    from torch_geometric.data import Data

    # 重新导入（spawn模式下子进程需要重新import）
    device = torch.device(f"cuda:{gpu_id}")
    torch.cuda.set_device(gpu_id)

    # 重建data对象
    data = Data(
        x=data_dict["x"],
        edge_index=data_dict["edge_index"],
        y=data_dict["y"],
        train_mask=data_dict["train_mask"],
        val_mask=data_dict["val_mask"],
        test_mask=data_dict["test_mask"],
    )

    print(f"[GPU {gpu_id}] 工作进程已启动")

    while True:
        task = task_queue.get()
        if task is None:  # 终止信号
            break

        idx, params = task
        try:
            print(f"[GPU {gpu_id}] 开始任务 {idx}: {params}")
            result = _train_single_config(
                model_name, params, data, device, class_names_list
            )
            result_queue.put((idx, result, None))
            print(
                f"[GPU {gpu_id}] 完成任务 {idx} | Val Acc: {result['best_val_acc']:.4f}"
            )
        except Exception as e:
            result_queue.put((idx, None, str(e)))
            print(f"[GPU {gpu_id}] 任务 {idx} 失败: {e}")
        finally:
            # 清理GPU显存
            torch.cuda.empty_cache()
            gc.collect()


def _train_single_config(model_name, params, data, device, class_names_list):
    """在指定device上训练单个参数组合，返回结果dict"""
    import torch.optim as optim

    in_dim = data.num_features
    out_dim = len(class_names_list)

    # 根据model_name构建模型
    if model_name == "MLP":
        model = MLPWithDropout(
            in_dim,
            params["hidden_dim"],
            out_dim,
            params["dropout"],
            params["num_layers"],
        )
    elif model_name == "GCN":
        model = GCN(
            in_dim,
            params["hidden_dim"],
            out_dim,
            params["dropout"],
            params["num_layers"],
        )
    elif model_name == "GraphSAGE":
        model = GraphSAGEWithNorm(
            in_dim,
            params["hidden_dim"],
            out_dim,
            params["dropout"],
            params["num_layers"],
        )
    elif model_name == "GAT":
        model = GAT(
            in_dim,
            params["hidden_dim"],
            out_dim,
            params["heads"],
            params["dropout"],
            params["num_layers"],
            params["use_skip"],
        )
    elif model_name == "GATv2":
        model = GATv2Model(
            in_dim,
            params["hidden_dim"],
            out_dim,
            params["heads"],
            params["dropout"],
            params["num_layers"],
        )
    else:
        raise ValueError(f"未知模型: {model_name}")

    num_params = sum(p.numel() for p in model.parameters())
    optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

    trained_model, train_losses, val_losses, train_accs, val_accs, best_val_acc = (
        train_model(
            model, data, device, optimizer, epochs=500, patience=50, verbose=False
        )
    )

    eval_results = evaluate_model(trained_model, data, device)

    config = {
        **params,
        "best_val_acc": best_val_acc,
        "num_params": num_params,
        **eval_results,
    }
    training_history = {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_accs": train_accs,
        "val_accs": val_accs,
    }

    # 保存（文件IO在子进程中执行）
    save_model_and_results(trained_model, config, model_name, training_history)
    save_detailed_results(
        model_name,
        config,
        training_history,
        eval_results["test_true"],
        eval_results["test_pred"],
        class_names_list,
    )

    del trained_model, model, optimizer
    return config


# def run_parallel_experiments(
#     model_name, param_combinations_list, data, class_names_list, num_gpus=None
# ):
#     """
#     多GPU并行参数搜索主函数（任务并行：每GPU跑一个参数组合）

#     参数:
#         model_name: 模型名称字符串
#         param_combinations_list: list of dict，每个dict是一组参数
#         data: PyG Data对象（在主进程中）
#         class_names_list: 类别名称列表
#         num_gpus: 使用的GPU数量（默认使用全部）

#     返回:
#         results: list of config dicts，按验证准确率排序
#     """
#     if num_gpus is None:
#         num_gpus = NUM_GPUS
#     num_gpus = max(1, min(num_gpus, NUM_GPUS))

#     total = len(param_combinations_list)
#     print(f"\n{'='*60}")
#     print(f"开始 {model_name} 参数搜索")
#     print(f"总任务数: {total} | 使用GPU数: {num_gpus} | 理论加速比: ~{num_gpus}x")
#     print(f"{'='*60}")

#     # 将data转为可序列化的dict（spawn模式需要序列化）
#     data_dict = {
#         "x": data.x,
#         "edge_index": data.edge_index,
#         "y": data.y,
#         "train_mask": data.train_mask,
#         "val_mask": data.val_mask,
#         "test_mask": data.test_mask,
#     }

#     # 创建任务队列和结果队列
#     ctx = mp.get_context("spawn")
#     task_queue = ctx.Queue()
#     result_queue = ctx.Queue()

#     # 填充任务队列
#     for idx, params in enumerate(param_combinations_list):
#         task_queue.put((idx, params))

#     # 每个GPU发送一个终止信号
#     for _ in range(num_gpus):
#         task_queue.put(None)

#     # 启动工作进程（每个GPU一个进程）
#     processes = []
#     for gpu_id in range(num_gpus):
#         p = ctx.Process(
#             target=_worker_process,
#             args=(
#                 gpu_id,
#                 task_queue,
#                 result_queue,
#                 data_dict,
#                 model_name,
#                 class_names_list,
#             ),
#             daemon=True,
#         )
#         p.start()
#         processes.append(p)

#     # 收集结果
#     results_dict = {}
#     completed = 0
#     while completed < total:
#         idx, result, error = result_queue.get()
#         completed += 1
#         if error:
#             print(f"  ✗ 任务 {idx} 失败: {error}")
#         else:
#             results_dict[idx] = result
#             print(
#                 f"  [{completed}/{total}] ✓ {model_name} 任务{idx} "
#                 f"Val={result['best_val_acc']:.4f} Test={result['test_acc']:.4f}"
#             )

#     # 等待所有进程结束
#     for p in processes:
#         p.join()

#     # 按原始顺序整理结果
#     results = [results_dict[i] for i in range(total) if i in results_dict]

#     if results:
#         best = max(results, key=lambda x: x["best_val_acc"])
#         print(f"\n{model_name} 最佳模型: {best}")

#     return results
# ============================================================
# 多GPU并行调度函数（从磁盘import worker，解决spawn问题）
# ============================================================
import sys, importlib
import torch.multiprocessing as mp

# 确保能找到gnn_worker.py
if "." not in sys.path:
    sys.path.insert(0, ".")

# 重新加载（防止多次运行Cell时用到旧版本）
if "gnn_worker" in sys.modules:
    importlib.reload(sys.modules["gnn_worker"])

from gnn_worker import worker_process  # ← 从.py文件import，spawn子进程能找到它


def run_parallel_experiments(model_name, param_combinations_list, data, class_names_list, num_gpus=None):
    """
    多GPU并行参数搜索（任务并行：每GPU同时跑一个参数组合）
    """
    if num_gpus is None:
        num_gpus = torch.cuda.device_count()
    num_gpus = max(1, min(num_gpus, torch.cuda.device_count()))

    total = len(param_combinations_list)
    print(f"\n{'='*60}")
    print(f"开始 {model_name} 参数搜索")
    print(f"总任务数: {total} | 使用GPU数: {num_gpus} | 理论加速: ~{num_gpus}x")
    print(f"{'='*60}")

    # data序列化为dict（spawn模式需要pickle传递）
    data_dict = {
        "x":          data.x.cpu(),
        "edge_index": data.edge_index.cpu(),
        "y":          data.y.cpu(),
        "train_mask": data.train_mask.cpu(),
        "val_mask":   data.val_mask.cpu(),
        "test_mask":  data.test_mask.cpu(),
    }

    ctx          = mp.get_context("spawn")
    task_queue   = ctx.Queue()
    result_queue = ctx.Queue()

    # 填充任务
    for idx, params in enumerate(param_combinations_list):
        task_queue.put((idx, params))
    # 终止信号（每个GPU一个）
    for _ in range(num_gpus):
        task_queue.put(None)

    # 启动进程
    processes = []
    for gpu_id in range(num_gpus):
        p = ctx.Process(
            target=worker_process,   # ← 来自gnn_worker.py，spawn能正确找到
            args=(gpu_id, task_queue, result_queue, data_dict, model_name, class_names_list),
            daemon=True,
        )
        p.start()
        processes.append(p)

    # 收集结果
    results_dict = {}
    completed    = 0
    while completed < total:
        idx, result, error = result_queue.get()
        completed += 1
        if error:
            print(f"  ✗ 任务 #{idx} 失败:\n{error}")
        else:
            results_dict[idx] = result
            print(f"  [{completed}/{total}] ✓ {model_name} #{idx} "
                  f"Val={result['best_val_acc']:.4f} Test={result['test_acc']:.4f}")

    for p in processes:
        p.join()

    results = [results_dict[i] for i in range(total) if i in results_dict]
    if results:
        best = max(results, key=lambda x: x["best_val_acc"])
        print(f"\n{model_name} 最佳: {best}")
    return results


print("✓ run_parallel_experiments 已定义")


7. 定义训练和评估函数（多GPU版）
✓ run_parallel_experiments 已定义


In [7]:
# ============================================================
# Code eleven: 参数网格配置
# ============================================================
print("\n" + "=" * 60)
print("8. 模型参数实验配置")
print("=" * 60)

in_dim = data.num_features
out_dim = num_classes

param_grids = {
    "MLP": {
        "hidden_dim": [128, 256],
        "dropout": [0.2, 0.25, 0.3],
        "num_layers": [2, 3, 4],
    },
    "GCN": {
        "hidden_dim": [128, 256],
        "dropout": [0.2, 0.25, 0.3],
        "num_layers": [2, 3, 4],
    },
    "GraphSAGE": {
        "hidden_dim": [128, 256],
        "dropout": [0.2, 0.25, 0.3],
        "num_layers": [2, 3, 4],
    },
    "GAT": {
        "hidden_dim": [64, 128],
        "heads": [4, 8],
        "dropout": [0.2, 0.25, 0.3],
        "num_layers": [2, 3],
        "use_skip": [True],
    },
    "GATv2": {
        "hidden_dim": [64, 128],
        "heads": [4, 8],
        "dropout": [0.2, 0.25, 0.3],
        "num_layers": [2, 3],
    },
}
print("参数网格配置完成")


# ============================================================
# ★ 辅助函数：把参数网格展开为 list of dict
# ============================================================
def expand_param_grid(grid):
    """将param_grid展开为 list of dict，便于并行分发"""
    keys = list(grid.keys())
    values = list(grid.values())
    return [dict(zip(keys, combo)) for combo in product(*values)]


8. 模型参数实验配置
参数网格配置完成


In [ ]:
# ============================================================
# Code twelve: 9.1 MLP 多GPU并行实验
# ============================================================
print("\n" + "=" * 50)
print("9.1 MLP 参数实验（多GPU并行）")
print("=" * 50)

mlp_combos = expand_param_grid(param_grids["MLP"])
mlp_results = run_parallel_experiments("MLP", mlp_combos, data, class_names)

if mlp_results:
    best_mlp = max(mlp_results, key=lambda x: x["best_val_acc"])
    print(
        f"\nMLP 最佳: hidden_dim={best_mlp['hidden_dim']}, dropout={best_mlp['dropout']}, "
        f"num_layers={best_mlp['num_layers']}, Test Acc={best_mlp['test_acc']:.4f}"
    )


9.1 MLP 参数实验（多GPU并行）

开始 MLP 参数搜索
总任务数: 18 | 使用GPU数: 8 | 理论加速: ~8x


In [ ]:
# ============================================================
# Code thirteen: 9.2 GCN 多GPU并行实验
# ============================================================
print("\n" + "=" * 50)
print("9.2 GCN 参数实验（多GPU并行）")
print("=" * 50)

gcn_combos = expand_param_grid(param_grids["GCN"])
gcn_results = run_parallel_experiments("GCN", gcn_combos, data, class_names)

if gcn_results:
    best_gcn = max(gcn_results, key=lambda x: x["best_val_acc"])
    print(
        f"\nGCN 最佳: hidden_dim={best_gcn['hidden_dim']}, dropout={best_gcn['dropout']}, "
        f"num_layers={best_gcn['num_layers']}, Test Acc={best_gcn['test_acc']:.4f}"
    )

In [ ]:
# ============================================================
# Code fourteen/fifteen: 9.3 GraphSAGE 多GPU并行实验
# ============================================================
print("\n" + "=" * 50)
print("9.3 GraphSAGE 参数实验（多GPU并行）")
print("=" * 50)

sage_combos = expand_param_grid(param_grids["GraphSAGE"])
sage_results = run_parallel_experiments("GraphSAGE", sage_combos, data, class_names)

if sage_results:
    best_sage = max(sage_results, key=lambda x: x["best_val_acc"])
    print(
        f"\nGraphSAGE 最佳: hidden_dim={best_sage['hidden_dim']}, dropout={best_sage['dropout']}, "
        f"num_layers={best_sage['num_layers']}, Test Acc={best_sage['test_acc']:.4f}"
    )

In [ ]:
# ============================================================
# Code sixteen/seventeen: 9.4 GATv2 多GPU并行实验
# （原来在CPU跑，现在全部上GPU！）
# ============================================================
print("\n" + "=" * 50)
print("9.4 GATv2 参数实验（多GPU并行，原CPU→现GPU）")
print("=" * 50)

gatv2_combos = expand_param_grid(param_grids["GATv2"])
gatv2_results = run_parallel_experiments("GATv2", gatv2_combos, data, class_names)

if gatv2_results:
    best_gatv2 = max(gatv2_results, key=lambda x: x["best_val_acc"])
    print(
        f"\nGATv2 最佳: hidden_dim={best_gatv2['hidden_dim']}, heads={best_gatv2['heads']}, "
        f"dropout={best_gatv2['dropout']}, num_layers={best_gatv2['num_layers']}, "
        f"Test Acc={best_gatv2['test_acc']:.4f}"
    )

In [ ]:
# ============================================================
# Code eighteen: 9.5 GAT 多GPU并行实验
# （原来在CPU跑，现在全部上GPU！）
# ============================================================
print("\n" + "=" * 50)
print("9.5 GAT 参数实验（多GPU并行，原CPU→现GPU）")
print("=" * 50)

gat_combos = expand_param_grid(param_grids["GAT"])
gat_results = run_parallel_experiments("GAT", gat_combos, data, class_names)

if gat_results:
    best_gat = max(gat_results, key=lambda x: x["best_val_acc"])
    print(
        f"\nGAT 最佳: hidden_dim={best_gat['hidden_dim']}, heads={best_gat['heads']}, "
        f"dropout={best_gat['dropout']}, num_layers={best_gat['num_layers']}, "
        f"use_skip={best_gat['use_skip']}, Test Acc={best_gat['test_acc']:.4f}"
    )

In [ ]:
# ============================================================
# Code nineteen: 10. 结果汇总和可视化（不变）
# ============================================================
print("\n" + "=" * 60)
print("10. 结果汇总和可视化")
print("=" * 60)

all_best_models = {}
for model_name, results in [
    ("MLP", mlp_results),
    ("GCN", gcn_results),
    ("GraphSAGE", sage_results),
    ("GAT", gat_results),
    ("GATv2", gatv2_results),
]:
    if results:
        all_best_models[model_name] = max(results, key=lambda x: x["best_val_acc"])

if all_best_models:
    summary_data = []
    for model_name, best_model in all_best_models.items():
        summary_data.append(
            {
                "Model": model_name,
                "Best Val Acc": best_model["best_val_acc"],
                "Test Acc": best_model["test_acc"],
                "Test F1 (Macro)": best_model["test_f1_macro"],
                "Test F1 (Weighted)": best_model["test_f1_weighted"],
                "Precision (Macro)": best_model.get("test_precision_macro", 0),
                "Recall (Macro)": best_model.get("test_recall_macro", 0),
                "Hidden Dim": best_model["hidden_dim"],
                "Dropout": best_model["dropout"],
                "Layers": best_model["num_layers"],
                "Heads": best_model.get("heads", "N/A"),
                "Params": best_model.get("num_params", 0),
            }
        )

    summary_df = pd.DataFrame(summary_data).sort_values("Test Acc", ascending=False)
    print("\n模型性能对比表:")
    print(summary_df.to_string(index=False))

    summary_df.to_csv(os.path.join(RESULTS_DIR, "model_summary.csv"), index=False)
    print(f"\n✓ 汇总表已保存到 {RESULTS_DIR}")

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    ax1 = axes[0, 0]
    bars = ax1.bar(summary_df["Model"], summary_df["Test Acc"], color="steelblue")
    ax1.set_ylabel("Accuracy")
    ax1.set_title("Model Test Accuracy Comparison")
    ax1.set_ylim(0, 1)
    for bar, acc in zip(bars, summary_df["Test Acc"]):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{acc:.4f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    x = np.arange(len(summary_df))
    w = 0.35
    ax2.bar(
        x - w / 2,
        summary_df["Test F1 (Macro)"],
        w,
        label="Macro F1",
        color="forestgreen",
    )
    ax2.bar(
        x + w / 2,
        summary_df["Test F1 (Weighted)"],
        w,
        label="Weighted F1",
        color="lightcoral",
    )
    ax2.set_xticks(x)
    ax2.set_xticklabels(summary_df["Model"], rotation=45, ha="right")
    ax2.set_ylabel("F1 Score")
    ax2.set_title("Model F1 Score Comparison")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    ax3.scatter(summary_df["Params"] / 1000, summary_df["Test Acc"], s=100, alpha=0.6)
    for _, row in summary_df.iterrows():
        ax3.annotate(
            row["Model"],
            (row["Params"] / 1000, row["Test Acc"]),
            fontsize=8,
            ha="center",
        )
    ax3.set_xlabel("Number of Parameters (K)")
    ax3.set_ylabel("Test Accuracy")
    ax3.set_title("Model Efficiency: Parameters vs Accuracy")
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    metrics = ["Test Acc", "Test F1 (Macro)", "Precision (Macro)", "Recall (Macro)"]
    num_metrics = len(metrics)
    angles = np.linspace(0, 2 * np.pi, num_metrics, endpoint=False).tolist()
    angles += angles[:1]
    for _, row in summary_df.iterrows():
        values = [row[m] for m in metrics] + [row[metrics[0]]]
        ax4.plot(angles, values, "o-", linewidth=2, label=row["Model"], alpha=0.7)
    ax4.set_xticks(angles[:-1])
    ax4.set_xticklabels(metrics)
    ax4.set_ylim(0, 1)
    ax4.set_title("Multi-metric Radar Chart")
    ax4.legend(loc="upper right", bbox_to_anchor=(1.3, 1.0))
    ax4.grid(True)

    plt.suptitle(
        "Comprehensive Model Performance Comparison", fontsize=14, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(FIGURE_PATHS["test_comparison"], dpi=200, bbox_inches="tight")
    plt.show()

    best_model_name = summary_df.iloc[0]["Model"]
    print(f"\n🏆 最终最佳模型: {best_model_name}")
    print(f"   测试准确率: {summary_df.iloc[0]['Test Acc']:.4f}")
    print(f"   Macro F1: {summary_df.iloc[0]['Test F1 (Macro)']:.4f}")

print("\n" + "=" * 60)
print("实验完成！")
print(f"所有结果已保存到: {BASE_DIR}")
print(f"  - 模型文件: {MODELS_DIR}")
print(f"  - 训练曲线: {CURVES_DIR}")
print(f"  - 混淆矩阵: {CONFUSION_DIR}")
print(f"  - 结果汇总: {RESULTS_DIR}")
print("=" * 60)